Importing Libraries

In [4]:
%pip install --force-reinstall matplotlib numpy

import requests
import astropy
import numpy as np
import urllib.request
import json
import geocoder
import webbrowser
import time
import geopy
import matplotlib.pyplot as plt

  You can safely remove it manually.
  You can safely remove it manually.
  You can safely remove it manually.
error: uninstall-no-record-file

× Cannot uninstall matplotlib 3.10.3
╰─> The package's contents are unknown: no RECORD file was found for matplotlib.

hint: You might be able to recover from this via: pip install --force-reinstall --no-deps matplotlib==3.10.3


  Using cached matplotlib-3.10.3-cp312-cp312-win_amd64.whl.metadata (11 kB)
  Using cached numpy-2.3.1-cp312-cp312-win_amd64.whl.metadata (60 kB)
  Using cached contourpy-1.3.2-cp312-cp312-win_amd64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached fonttools-4.59.0-cp312-cp312-win_amd64.whl.metadata (110 kB)
  Using cached kiwisolver-1.4.8-cp312-cp312-win_amd64.whl.metadata (6.3 kB)
  Using cached packaging-25.0-py3-none-any.whl.metadata (3.3 kB)
  Using cached pillow-11.3.0-cp312-cp312-win_amd64.whl.metadata (9.2 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached matplotlib-3.10.3-cp312-cp312-win_amd64.whl (8.1 MB)
Using cached numpy-2.3.1-cp312-cp312-win_amd64.whl (12.7 MB)
Using cached contourpy-1.3.2-cp312-cp312-win_amd64.whl (223 kB)
Using cached cycler-0

ImportError: DLL load failed while importing _path: The specified module could not be found.

Fetching ISS (Two line element) TLE data

In [2]:

response = urllib.request.urlopen('http://api.open-notify.org/iss-now.json')
result = json.loads(response.read())
print(result)


{'timestamp': 1753301978, 'message': 'success', 'iss_position': {'latitude': '-6.1967', 'longitude': '63.5640'}}


In [3]:
from datetime import timedelta, datetime
from geopy.geocoders import Nominatim
import pytz
ist = pytz.timezone('Asia/Kolkata')

# Initialize Nominatim API
geolocator = Nominatim(user_agent="my_geopy_app")
name = str(input("Enter a location name: "))
location = geolocator.geocode(name)
print("\nLocation Coordinates:")
print(f"Latitude: {location.latitude}")
print(f"Longitude: {location.longitude}\n")


latitude, longitude = location.latitude, location.longitude
from skyfield.api import EarthSatellite, Topos, load

# Example TLE lines
line1 = '1 25544U 98067A   25204.16864096  .00010724  00000+0  19544-3 0  9992'
line2 = '2 25544  51.6344 128.7280 0002115 106.2807 253.8414 15.50025731520740'
satellite = EarthSatellite(line1, line2, 'ISS (ZARYA)')

# Load timescale and observer location
ts = load.timescale()
observer = Topos(latitude_degrees=latitude, longitude_degrees=longitude)
t0 = ts.now()
t1 = ts.utc(t0.utc_datetime() + timedelta(days=0.5))  # Next 12 hours

# Find events (rise, culmination, set)
times, events = satellite.find_events(observer, t0, t1, altitude_degrees=0.0) 

event_names = ['rise', 'culminate', 'set']
print("Next ISS Passes:\n")
group = []
for ti, event in zip(times, events):
    utc_dt = ti.utc_datetime().replace(tzinfo=pytz.utc)
    ist_dt = utc_dt.astimezone(ist)
    group.append(f"{event_names[event].capitalize()}: {ist_dt.strftime('%Y-%m-%d %H:%M:%S IST')}")
    if event == 2:  # 'set' event
        print("\n".join(group))
        print("-" * 40)
        group = []


Location Coordinates:
Latitude: 21.2380912
Longitude: 81.6336993

Next ISS Passes:

Rise: 2025-07-24 11:51:39 IST
Culminate: 2025-07-24 11:55:12 IST
Set: 2025-07-24 11:58:46 IST
----------------------------------------
Rise: 2025-07-24 13:26:10 IST
Culminate: 2025-07-24 13:31:33 IST
Set: 2025-07-24 13:36:56 IST
----------------------------------------


In [ ]:
from mpl_toolkits.basemap import Basemap

In [ ]:

# llcrnrlat,llcrnrlon,urcrnrlat,urcrnrlon
# are the lat/lon values of the lower left and upper right corners of the map.
# resolution = 'c' means use crude resolution coastlines.

# Load timescale
ts = load.timescale()
now = ts.now()
minutes = np.arange(0, 91, 1)
times = ts.utc(now.utc_datetime() + np.array([np.timedelta64(int(m), 'm') for m in minutes]))
# ISS subpoint coordinates
lats = []
lons = []
for t in times:
    geocentric = satellite.at(t)
    subpoint = geocentric.subpoint()
    lats.append(subpoint.latitude.degrees)  
    lons.append(subpoint.longitude.degrees)
m = Basemap(projection='cyl',llcrnrlat=-90,urcrnrlat=90,\
             llcrnrlon=-180,urcrnrlon=180,resolution='c')
m.drawcoastlines()
m.bluemarble()
m.fillcontinents(color='grey')
# draw parallels and meridians.
m.drawparallels(np.arange(-90.,91.,30.))
m.drawmeridians(np.arange(-180.,181.,60.))
m.drawmapboundary(fill_color='grey')
plt.title("Cylindrical Equal-Area Projection")
plt.show()

ImportError: DLL load failed while importing _path: The specified module could not be found.